In [7]:
pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [9]:
import torch

# Example: 2 objects in an image
boxes = torch.tensor([[10, 20, 100, 150], [200, 250, 300, 400]], dtype=torch.float32)  # (xmin, ymin, xmax, ymax)
labels = torch.tensor([0, 2], dtype=torch.int64)  # class indices for each object

target = {
    'boxes': boxes,
    'labels': labels
}

In [10]:
import os
import torch
from PIL import Image
from torchvision.transforms import functional as F

class WasteDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, label_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transforms = transforms
        self.images = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.images[idx])
        label_path = os.path.join(self.label_dir, self.images[idx].replace('.jpg', '.txt'))

        img = Image.open(img_path).convert("RGB")
        width, height = img.size

        boxes = []
        labels = []

        with open(label_path, 'r') as f:
            for line in f.readlines():
                cls, x_center, y_center, w, h = map(float, line.strip().split())
                xmin = (x_center - w/2) * width
                xmax = (x_center + w/2) * width
                ymin = (y_center - h/2) * height
                ymax = (y_center + h/2) * height
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(int(cls))  # class ID

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}

        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.images)

In [11]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def get_model(num_classes):
    # Load pre-trained Faster R-CNN
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)

    # Replace classifier with your own
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

In [ ]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


class WasteDataset(Dataset):
    def __init__(self, img_dir, label_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transforms = transforms
        self.images = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.images[idx])
        label_path = os.path.join(self.label_dir, self.images[idx].replace('.jpg', '.txt'))

        img = Image.open(img_path).convert("RGB")
        width, height = img.size

        boxes = []
        labels = []

        with open(label_path, 'r') as f:
            for line in f.readlines():
                cls, x_center, y_center, w, h = map(float, line.strip().split())
                xmin = (x_center - w/2) * width
                xmax = (x_center + w/2) * width
                ymin = (y_center - h/2) * height
                ymax = (y_center + h/2) * height
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(int(cls))

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}

        if self.transforms:
            img = self.transforms(img)
        else:
            img = transforms.ToTensor()(img)

        return img, target

    def __len__(self):
        return len(self.images)


def get_model(num_classes):
    model = fasterrcnn_resnet50_fpn(pretrained=True)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


# Training config
img_dir = r"C:\Users\Alfina\Desktop\major\Warp-D\train\images"
label_dir = r"C:\Users\Alfina\Desktop\major\Warp-D\train\labels"
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
dataset = WasteDataset(img_dir, label_dir)
data_loader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))

model = get_model(num_classes=29)  # 28 classes + 1 background
model.to(device)
model.train()

optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
num_epochs = 10

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    for imgs, targets in data_loader:
        imgs = list(img.to(device) for img in imgs)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        print(f"Loss: {losses.item():.4f}")

# Save model
torch.save(model.state_dict(), r"C:\wasteprojectfaster_rcnn_waste.pth")


C:\Users\Alfina\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Alfina\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to C:\Users\Alfina/.cache\torch\hub\checkpoints\fasterrcnn_r

Epoch 1/10
Loss: 3.2810
Loss: 2.2060
Loss: 2.0448
Loss: 1.3039
Loss: 0.4470
Loss: 0.6844
Loss: 1.4413
Loss: 0.7659
Loss: 0.3880
Loss: 0.4452
Loss: 0.9410
Loss: 0.2489
Loss: 1.5558
Loss: 0.7217
Loss: 1.2064
Loss: 0.4961
Loss: 0.5506
Loss: 0.9550
Loss: 0.4568
Loss: 2.1744
Loss: 1.5056
Loss: 1.3288
Loss: 1.0696
Loss: 0.9472
Loss: 1.4698
Loss: 0.9661
Loss: 1.0080
Loss: 0.8789
Loss: 0.9881
Loss: 1.3369
Loss: 1.8773
Loss: 1.4933
Loss: 0.7210
Loss: 1.0094
Loss: 0.5204
Loss: 0.3334
Loss: 0.9745
Loss: 0.6036
Loss: 0.4056
Loss: 1.1352
Loss: 1.5338
Loss: 0.8536
Loss: 0.9844
Loss: 1.2885
Loss: 0.5130
Loss: 0.9077
Loss: 1.6839
Loss: 1.3006
Loss: 1.2555
Loss: 0.4103
Loss: 2.0521
Loss: 0.8621
Loss: 0.7666
Loss: 0.5887
Loss: 0.5129
Loss: 1.0611
Loss: 0.9428
Loss: 1.4327
Loss: 1.7797
Loss: 0.7645
Loss: 1.5055
Loss: 0.4519
Loss: 0.9027
Loss: 1.0788
Loss: 1.1831
Loss: 0.8684
Loss: 1.2232
Loss: 1.1505
Loss: 1.9324
Loss: 0.4632
Loss: 0.3645
Loss: 0.7150
Loss: 1.1314
Loss: 0.3573
Loss: 0.7551
Loss: 0.6732
L